# Facility-held-out expense regression and classification

**Target:** one prespecified comparable reporting measure, `TOT_OP_EXP` (total operating expenses, nominal source dollars). Predictors summarize the same report period, so this is an exploratory association task for unseen facilities, not a future forecast.

Classification asks whether reported expense exceeds the training median for this measure. The cutoff has no clinical, demand, or efficiency interpretation. Neither `Amount`, facility/name IDs, ordinal category codes, nor derived clusters enter the predictors. COUNTY uses training-fit one-hot encoding with unseen categories ignored.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hospital_ml").is_dir())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Image
from hospital_ml.data import read_raw, prepare_data, write_json
SEED = 42


In [2]:
from hospital_ml.supervised import run_supervised
frame, audit = prepare_data(read_raw())
metrics = run_supervised(frame, ROOT / "reports/supervised", seed=SEED, target_variable="TOT_OP_EXP")
print("Measure:", metrics["target_description"], "Units:", metrics["target_units"])
print("Rows:", metrics["rows"], "Facilities:", metrics["facilities"])
print("Development-training expense threshold:", metrics["development_training_expense_threshold"])
print("Regression choice:", metrics["regression_selection"])
print("Classification choice:", metrics["classification_selection"])

Measure: Reported total operating expenses Units: Source-reported dollars; reporting-period totals, not inflation adjusted
Rows: {'development': 2154, 'holdout': 536} Facilities: {'development': 377, 'holdout': 95, 'overlap': 0}
Development-training expense threshold: 86796390.5
Regression choice: {'criterion': 'minimum mean development-fold MAE, including dummy baselines', 'selected_model': 'random_forest'}
Classification choice: {'criterion': 'maximum mean development-fold balanced accuracy, including dummy baseline', 'selected_model': 'random_forest'}


## Frozen evaluation protocol

First hold out 20% of facilities. Use three GroupKFold development folds to select among fixed model configurations. Numeric imputation, optional 1%/99% feature clipping, robust scaling, missing indicators, and categorical encoding are fitted within each training fold. The classification expense threshold is recomputed from each fold's training targets; it is then frozen from all development targets for held-out evaluation.

Regression compares mean/median dummy baselines, Ridge, and bounded Random Forest; select by mean development MAE. Classification compares a most-frequent dummy, logistic regression, and bounded Random Forest; select by mean development balanced accuracy. No hyperparameter search on held-out data is performed. Evaluate only baselines and the selected model on the holdout. All targets, including extremes and zeros, remain in evaluation.

In [3]:
print("Development CV results")
display(pd.DataFrame(metrics["regression_cv_mean"]).T)
display(pd.DataFrame(metrics["classification_cv_mean"]).T)
print("Fold-specific training expense cutoffs:", metrics["classification_cv_training_thresholds"])
print("Development sensitivity without clipping:", metrics["development_sensitivity_without_clipping"])

Development CV results


,mae,rmse,r2,median_absolute_error
dummy_mean,2.651740e+08,5.812006e+08,-0.048739,2.023826e+08
dummy_median,2.200752e+08,5.985226e+08,-0.106145,7.686681e+07
random_forest,1.728199e+08,5.731614e+08,0.035726,4.864126e+07
ridge,2.113700e+08,5.834451e+08,-0.017727,8.563195e+07


,accuracy,balanced_accuracy,macro_f1,roc_auc,pr_auc_average_precision
dummy_most_frequent,0.505107,0.500000,0.334969,0.500000,0.494893
logistic_regression,0.794800,0.789896,0.790509,0.861567,0.836704
random_forest,0.807799,0.806027,0.806101,0.868780,0.850359


Fold-specific training expense cutoffs: {1: 93163400.0, 2: 87170321.0, 3: 81132287.0}
Development sensitivity without clipping: {'regression_mean_mae': 154641242.6424133, 'classification_mean_balanced_accuracy': 0.804381321308627, 'used_for_selection': False}


In [4]:
print("Held-out regression: absolute dollar errors, R², and equal-facility sensitivity")
display(pd.read_csv(ROOT / "reports/supervised/regression_holdout.csv"))
print("Held-out classification: compare PR average precision against reported prevalence")
display(pd.read_csv(ROOT / "reports/supervised/classification_holdout.csv"))
chosen = metrics["classification_selection"]["selected_model"]
confusion = metrics["classification_holdout"][chosen]["confusion_matrix_labels_0_1"]
display(pd.DataFrame(confusion, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]))

Held-out regression: absolute dollar errors, R², and equal-facility sensitivity


,model,mae,rmse,r2,median_absolute_error,facility_macro_mae
0,dummy_mean,1.851955e+08,2.134324e+08,-0.118151,1.960623e+08,1.887684e+08
1,dummy_median,1.508634e+08,2.228653e+08,-0.219172,8.334865e+07,1.469916e+08
2,random_forest,1.002225e+08,1.665206e+08,0.319363,4.976571e+07,9.582409e+07


Held-out classification: compare PR average precision against reported prevalence


,model,accuracy,balanced_accuracy,macro_f1,precision,recall,roc_auc,pr_auc_average_precision,positive_prevalence,facility_macro_accuracy
0,dummy_most_frequent,0.486940,0.500000,0.327478,0.00000,0.000000,0.500000,0.513060,0.51306,0.516855
1,random_forest,0.789179,0.789669,0.789178,0.80916,0.770909,0.849725,0.811969,0.51306,0.791830


,Predicted 0,Predicted 1
Actual 0,211,50
Actual 1,63,212


In [5]:
print("95% whole-facility bootstrap intervals (conditional on the fitted model)")
display(pd.DataFrame(metrics["regression_selected_model_ci"]["intervals"], index=["2.5%", "97.5%"]).T)
display(pd.DataFrame(metrics["classification_selected_model_ci"]["intervals"], index=["2.5%", "97.5%"]).T)
for limitation in metrics["limitations"]:
    print("-", limitation)

95% whole-facility bootstrap intervals (conditional on the fitted model)


,2.5%,97.5%
mae,7.670381e+07,1.258220e+08
rmse,1.244781e+08,2.083720e+08
r2,-1.100254e-01,5.395759e-01
median_absolute_error,3.386454e+07,7.055211e+07


,2.5%,97.5%
accuracy,0.699176,0.869117
balanced_accuracy,0.700376,0.868195
macro_f1,0.693782,0.868245
precision,0.701839,0.914597
recall,0.648619,0.886714
roc_auc,0.767070,0.928773
pr_auc_average_precision,0.704458,0.928172
positive_prevalence,0.411877,0.603340


- Expense is a financial reporting measure, not a verified clinical or efficiency outcome.
- Facility holdout estimates transfer to unseen facilities in this historical extract; no temporal forecasting claim is made.
- Report periods and nominal dollar values differ; DAY_PER is included but there is no inflation or case-mix adjustment.
- The expense-category cutoff is an exploratory training median and has no clinical or policy interpretation.
- Facility bootstrap intervals are conditional on the fitted model and omit training and source-reporting uncertainty.
- Primary metrics and model selection weight reporting rows equally; facilities with more report periods have more influence. Facility-macro MAE/accuracy provide an equal-facility holdout sensitivity.
- Licensed beds are end-of-period counts; available and staffed beds are period averages. Their ratios are descriptive capacity features, not simultaneous utilization measures, and can exceed one.


RMSE is not divided by the single largest outcome to imply small practical error. MAE/RMSE are dollar errors for the chosen measure, with a median-error complement for skew. R² can be negative; the baseline comparison determines predictive value. Repeated years remain correlated within facilities, so uncertainty resamples whole facilities. These intervals omit training/source-reporting uncertainty.

Original mixed-`Variable` regression/classification results remain in the untouched notebooks and `reports/historical/original_saved_text_outputs.json`. They are not directly comparable to this changed target, cohort, validation, and preprocessing design. No numerical improvement over historical scores is claimed.